# 圣地巡礼Anitabi


数据 API 基础地址 https://api.anitabi.cn/

图片 API 基础地址 https://image.anitabi.cn/

In [43]:
import requests
from PIL import Image
from io import BytesIO


def get_metadata(id, request_type):
    if request_type == "litepoints":
        url = f"https://api.anitabi.cn/bangumi/{id}/lite"
    elif request_type == "anime":
        url = f"https://api.anitabi.cn/bangumi/{id}"
    elif request_type == "points_detail":
        url = f"https://api.anitabi.cn/bangumi/{id}/points/detail"
    response = requests.get(url)
    if response.status_code == 200:
        # 请求成功，打印返回的 JSON 数据
        data = response.json()
        return data
    else:
        print(f"Error: {response.status_code}")
        return None    

def get_images(img_url):
    response = requests.get(img_url)
    if response.status_code == 200:
        # 请求成功，将图像数据转换为 PIL 图像对象
        image = Image.open(BytesIO(response.content))
        # 显示图像
        return image
    else:
        print(f"Error: {response.status_code}")
        return None

In [154]:
import pandas as pd
import os
import time
from tqdm import tqdm
import numpy as np

# anime_id = 115908

def crawl(anime_id):
    metadata_savepath = "/data_nas/cehou/anitabi/metadata/"
    os.makedirs(os.path.join(metadata_savepath, f"{anime_id}"), exist_ok=True)
    litepoints_data = get_metadata(anime_id, "litepoints")
    litepoints_df = pd.DataFrame(litepoints_data['litePoints'])
    litepoints_df.to_csv(os.path.join(metadata_savepath, f"{anime_id}/litepoints_metadata.csv"), index=False)

    anime_data = get_metadata(anime_id, "anime")
    anime_df = pd.DataFrame([anime_data])
    anime_df.to_csv(os.path.join(metadata_savepath, f"{anime_id}/anime_metadata.csv"), index=False)

    detailed_points_data = get_metadata(anime_id, "points_detail")
    detailed_points_df = pd.DataFrame(detailed_points_data)
    detailed_points_df.to_csv(os.path.join(metadata_savepath, f"{anime_id}/detailed_points_metadata.csv"), index=False)
    print(f"Start crawling metadata for anime with ID: {anime_id}, Bangumi title: {anime_data['title']}, Total detailed points: {len(detailed_points_df)}")


    # 爬取图像
    if 'image' not in detailed_points_df.columns:
        print("No image column in detailed points metadata, skip crawling images")
        return
    
    anime_image_folder = os.path.join(metadata_savepath, f"{anime_id}/anime_imgs")
    os.makedirs(anime_image_folder, exist_ok=True)
    for i, row in tqdm(detailed_points_df.iterrows()):

        if row.image is np.nan:
            continue
        else:
            img_url = row.image.split('.jpg')[0] + '.jpg'

        anime_img_id = row.id
        anime_image = get_images(img_url)
        try:
            anime_image.save(os.path.join(anime_image_folder, f"{anime_img_id}.jpg"))
        except Exception as e:
            print(f"Error saving image with ID: {anime_img_id}, cannot write mode RGBA as JPEG")
        time.sleep(0.1)

anime_ids = [23686,92382,10639,215425,909,72942]
for anime_id in tqdm(anime_ids):
    crawl(anime_id)

  0%|          | 0/6 [00:00<?, ?it/s]

Start crawling metadata for anime with ID: 23686
Bangumi title: ソードアート・オンライン
Total detailed points: 16


 17%|█▋        | 1/6 [00:00<00:03,  1.32it/s]

No image column in detailed points metadata, skip crawling images
Start crawling metadata for anime with ID: 92382
Bangumi title: ソードアート・オンラインII
Total detailed points: 25


25it [00:14,  1.73it/s]
 33%|███▎      | 2/6 [00:15<00:37,  9.28s/it]

...

89it [00:53,  1.67it/s]
 67%|██████▋   | 4/6 [01:14<00:51, 25.51s/it]

Start crawling metadata for anime with ID: 909
Bangumi title: とらドラ!
Total detailed points: 28


28it [00:11,  2.38it/s]
 83%|████████▎ | 5/6 [01:26<00:20, 20.78s/it]

Start crawling metadata for anime with ID: 72942
Bangumi title: 中二病でも恋がしたい！戀
Total detailed points: 157


157it [01:30,  1.73it/s]
100%|██████████| 6/6 [02:58<00:00, 29.68s/it]


In [155]:
aaa = pd.read_csv("/data_nas/cehou/anitabi/metadata/23686/anime_metadata.csv")

In [161]:
from glob import glob
anitabi_dir = "/data_nas/cehou/anitabi/metadata/"
exists = [int(i) for i in os.listdir(anitabi_dir)]

In [163]:
for i in tqdm(range(1, 5)):
    if i in exists:
        continue
    else:
        try:
            crawl(i)
        except Exception as e:
            print(f"Error crawling anime with ID: {i}, {e}")
            continue

 50%|█████     | 2/4 [00:00<00:00,  7.90it/s]

Start crawling metadata for anime with ID: 1
Error: 404
Error crawling anime with ID: 1, 'NoneType' object is not subscriptable
Start crawling metadata for anime with ID: 2
Error: 404
Error crawling anime with ID: 2, 'NoneType' object is not subscriptable


100%|██████████| 4/4 [00:00<00:00,  8.80it/s]

Start crawling metadata for anime with ID: 3
Error: 404
Error crawling anime with ID: 3, 'NoneType' object is not subscriptable
Start crawling metadata for anime with ID: 4
Error: 404
Error crawling anime with ID: 4, 'NoneType' object is not subscriptable
